In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('bank-additional-full.csv', delimiter = ';')

In [3]:
cols_to_drop = ['duration', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
# at the same time, rename the columns so they are understandable. Please read the UCI page (https://archive.ics.uci.edu/ml/datasets/bank+marketing) for details
df = df.drop(columns=cols_to_drop).rename(columns={'job': 'job_type', 'default': 'default_status', 
                                                   'housing': 'housing_loan_status', 'loan': 'personal_loan_status', 
                                                   'contact': 'contact_type', 'month': 'contact_month', 
                                                   'day_of_week': 'contact_day_of_week', 'campaign': 'num_contacts', 
                                                   'pdays': 'days_last_contact', 'previous': 'previous_contacts', 
                                                   'poutcome': 'previous_outcome', 
                                                   'y': 'result'
                                                    })
# convert the target to numerical values
df['result'] = df['result'].replace({'yes': 1, 'no': 0})

D:\Temp\ipykernel_6516\2850889698.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['result'] = df['result'].replace({'yes': 1, 'no': 0})


In [4]:
df.head()


,age,job_type,marital,education,default_status,housing_loan_status,personal_loan_status,contact_type,contact_month,contact_day_of_week,num_contacts,days_last_contact,previous_contacts,previous_outcome,result
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   age                   41188 non-null  int64 
 1   job_type              41188 non-null  object
 2   marital               41188 non-null  object
 3   education             41188 non-null  object
 4   default_status        41188 non-null  object
 5   housing_loan_status   41188 non-null  object
 6   personal_loan_status  41188 non-null  object
 7   contact_type          41188 non-null  object
 8   contact_month         41188 non-null  object
 9   contact_day_of_week   41188 non-null  object
 10  num_contacts          41188 non-null  int64 
 11  days_last_contact     41188 non-null  int64 
 12  previous_contacts     41188 non-null  int64 
 13  previous_outcome      41188 non-null  object
 14  result                41188 non-null  int64 
dtypes: int64(5), object(10)
memory usage

In [6]:
df.isna().sum()

age                     0
job_type                0
marital                 0
education               0
default_status          0
housing_loan_status     0
personal_loan_status    0
contact_type            0
contact_month           0
contact_day_of_week     0
num_contacts            0
days_last_contact       0
previous_contacts       0
previous_outcome        0
result                  0
dtype: int64

In [7]:
df.result.value_counts()

result
0    36548
1     4640
Name: count, dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


x = df.drop(columns='result')
y = df['result']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
from category_encoders.target_encoder import TargetEncoder
from xgboost import XGBClassifier

estimators = [
    ('encoder', TargetEncoder()),
    ('clf', XGBClassifier(random_state=8))
]

pipe = Pipeline(estimators)

In [10]:
pipe.fit(x_train, y_train)

Pipeline(steps=[('encoder',
                 TargetEncoder(cols=['job_type', 'marital', 'education',
                                     'default_status', 'housing_loan_status',
                                     'personal_loan_status', 'contact_type',
                                     'contact_month', 'contact_day_of_week',
                                     'previous_outcome'])),
                ('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, d...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [11]:
from sklearn.metrics import accuracy_score

y_pred = pipe.predict(x_test)

accuracy_score(y_test, y_pred)

0.8959698956057296

In [12]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical


search_space = {
    'clf__n_estimators': Integer(50, 500),
    'clf__max_depth': Integer(3, 10),
    'clf__learning_rate': Real(0.01, 1.0, prior='log-uniform'),
    'clf__subsample': Real(0.1, 1.0, prior='uniform'),
    'clf__colsample_bytree': Real(0.1, 1.0, prior='uniform'),
}

In [13]:
opt = BayesSearchCV(
    pipe,
    search_space,
    cv = 10,
    n_iter = 20,
    scoring = 'accuracy',
    random_state = 8,
    n_jobs = -1
)

In [14]:
opt.fit(x_train, y_train)

BayesSearchCV(cv=10,
              estimator=Pipeline(steps=[('encoder',
                                         TargetEncoder(cols=['job_type',
                                                             'marital',
                                                             'education',
                                                             'default_status',
                                                             'housing_loan_status',
                                                             'personal_loan_status',
                                                             'contact_type',
                                                             'contact_month',
                                                             'contact_day_of_week',
                                                             'previous_outcome'])),
                                        ('clf',
                                         XGBClassifier(base_score=None,
                                                       booster=None,
                                                       callbacks=None,
                                                       colsample_bylevel=None,
                                                       colsample_bynod...
              search_spaces={'clf__colsample_bytree': Real(low=0.1, high=1.0, prior='uniform', transform='normalize'),
                             'clf__learning_rate': Real(low=0.01, high=1.0, prior='log-uniform', transform='normalize'),
                             'clf__max_depth': Integer(low=3, high=10, prior='uniform', transform='normalize'),
                             'clf__n_estimators': Integer(low=50, high=500, prior='uniform', transform='normalize'),
                             'clf__subsample': Real(low=0.1, high=1.0, prior='uniform', transform='normalize')})

In [15]:
opt.best_estimator_

Pipeline(steps=[('encoder',
                 TargetEncoder(cols=['job_type', 'marital', 'education',
                                     'default_status', 'housing_loan_status',
                                     'personal_loan_status', 'contact_type',
                                     'contact_month', 'contact_day_of_week',
                                     'previous_outcome'])),
                ('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=1.0, de...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.01,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=226, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [16]:
opt.best_params_

OrderedDict([('clf__colsample_bytree', 1.0),
             ('clf__learning_rate', 0.01),
             ('clf__max_depth', 3),
             ('clf__n_estimators', 226),
             ('clf__subsample', 1.0)])

In [17]:
opt.best_score_

np.float64(0.8981790591805765)

In [19]:
opt.predict_proba(x_test)

array([[0.9087094 , 0.09129061],
       [0.8904112 , 0.10958879],
       [0.7393273 , 0.2606727 ],
       ...,
       [0.89947504, 0.10052498],
       [0.95079625, 0.04920378],
       [0.92528296, 0.07471703]], shape=(8238, 2), dtype=float32)